## Data preprocessing

In [22]:
from keras.preprocessing.sequence import pad_sequences
import json
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from gensim.models import KeyedVectors


def preprocess_text(text, tokenizer, max_length):
    sequences = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(sequences, maxlen=max_length, padding='post')
    return padded[0] 

# Load JSON data
def load_data(filepath):
    with open(filepath, 'r') as file:
        data = json.load(file)
    return data

In [16]:

claims_data = load_data('data/train-claims.json')
evidence_data = load_data('data/evidence.json')

claims = [info['claim_text'] for info in claims_data.values()]
evidences = [evidence for evidence in evidence_data.values()]


In [5]:
# Load pre-trained word2vec
word_vectors = KeyedVectors.load('word2vec.wordvectors', mmap='r')

# Tokenize text data
tokenizer = Tokenizer()
tokenizer.fit_on_texts(claims + evidences)
vocab_size = len(tokenizer.word_index) + 1

# Convert texts to sequences of integers
claims_seq = tokenizer.texts_to_sequences(claims)
evidences_seq = tokenizer.texts_to_sequences(evidences)

# Pad sequences to ensure uniform length
max_length = max(max(len(seq) for seq in claims_seq), max(len(seq) for seq in evidences_seq))
claims_padded = pad_sequences(claims_seq, maxlen=max_length, padding='post')
evidences_padded = pad_sequences(evidences_seq, maxlen=max_length, padding='post')

## Creating the Embedding Matrix

In [6]:
# Create an embedding matrix
embedding_dim = 50  # dimension of word2vec vectors
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():
    if word in word_vectors:
        embedding_vector = word_vectors[word]
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector


## Building the LSTM Model

We will build a simple unidirectional LSTM model to compare claim and evidence embeddings.

In [11]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Dropout, concatenate
from tensorflow.keras import regularizers

# Define the model
def create_model():
        claims_input = Input(shape=(max_length,), dtype='int32')
        evidences_input = Input(shape=(max_length,), dtype='int32')

        # Shared Embedding layer
        embedding_layer = Embedding(vocab_size, embedding_dim, weights=[embedding_matrix], trainable=False)

        # LSTM layers
        claims_embeddings = embedding_layer(claims_input)
        evidences_embeddings = embedding_layer(evidences_input)

        claims_lstm = LSTM(64)(claims_embeddings)
        evidences_lstm = LSTM(16)(evidences_embeddings)

        # Concatenate and output
        concatenated = concatenate([claims_lstm, evidences_lstm])
        concatenated = Dense(64, kernel_regularizer=regularizers.l2(0.001), activation='relu')(concatenated)
        concatenated = Dropout(0.5)(concatenated)
        output = Dense(1, activation='sigmoid')(concatenated)  

    
        model = Model(inputs=[claims_input, evidences_input], outputs=output)
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

        return model

model = create_model()
print(model.summary())


Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_9 (InputLayer)        [(None, 278)]                0         []                            
                                                                                                  
 input_10 (InputLayer)       [(None, 278)]                0         []                            
                                                                                                  
 embedding_2 (Embedding)     (None, 278, 50)              528650    ['input_9[0][0]',             
                                                                     'input_10[0][0]']            
                                                                                                  
 lstm_4 (LSTM)               (None, 64)                   29440     ['embedding_2[0][0]']   

###  Training the Model

In [12]:
# Mock labels for training (you need actual labels here)
labels = np.random.randint(2, size=len(claims))

# Train the model
model.fit([claims_padded, evidences_padded], labels, epochs=10, batch_size=32, validation_split=0.1)


Epoch 1/10
35/35 [==============================] - 3s 57ms/step - loss: 0.7480 - accuracy: 0.5086 - val_loss: 0.7338 - val_accuracy: 0.5041
Epoch 2/10
35/35 [==============================] - 2s 53ms/step - loss: 0.7249 - accuracy: 0.5167 - val_loss: 0.7164 - val_accuracy: 0.5041
Epoch 3/10
35/35 [==============================] - 2s 56ms/step - loss: 0.7105 - accuracy: 0.5176 - val_loss: 0.7062 - val_accuracy: 0.5041
Epoch 4/10
35/35 [==============================] - 2s 59ms/step - loss: 0.7027 - accuracy: 0.5176 - val_loss: 0.7002 - val_accuracy: 0.5041
Epoch 5/10
35/35 [==============================] - 2s 49ms/step - loss: 0.6981 - accuracy: 0.5176 - val_loss: 0.6969 - val_accuracy: 0.5041
Epoch 6/10
35/35 [==============================] - 2s 58ms/step - loss: 0.6956 - accuracy: 0.5176 - val_loss: 0.6951 - val_accuracy: 0.5041
Epoch 7/10
35/35 [==============================] - 2s 54ms/step - loss: 0.6935 - accuracy: 0.5176 - val_loss: 0.6943 - val_accuracy: 0.5041
Epoch 8/10
35

## Test the model

In [23]:
claim_text = "The Earth’s climate sensitivity is so low that a doubling of atmospheric CO2 will result in a surface temperature change on the order of 1°C or less."
claim_seq = preprocess_text(claim_text, tokenizer, max_length)
claim_seq = claim_seq.squeeze() 

evidences_list = ["In his first paper on the matter, he estimated that global temperature would rise by around 5 to 6 °C (9.0 to 10.8 °F) if the quantity of CO 2 was doubled", "The 1990 IPCC First Assessment Report estimated that equilibrium climate sensitivity to a dou- bling of CO 2 lay between 1.5 and 4.5 °C (2.7 and 8.1 °F), with a \"best guess in the light of current knowledge\" of 2.5 °C (4.5 °F).", "John Bennet Lawes, English entrepreneur and agricultural scientist"] 
evidence_seqs = np.array([preprocess_text(ev, tokenizer, max_length) for ev in evidences_list])

claims_input = np.tile(claim_seq, (len(evidence_seqs), 1))

# Check shapes
print("Claims input shape:", claims_input.shape)  # Should be (number of evidences, max_length)
print("Evidences input shape:", evidence_seqs.shape)  # Should also be (number of evidences, max_length)

# Predict
predictions = model.predict([claims_input, evidence_seqs])

Claims input shape: (3, 278)
Evidences input shape: (3, 278)
1/1 [==============================] - 1s 502ms/step


In [25]:
threshold = 0.5  # This threshold can be adjusted based on your validation set performance or specific needs
related_evidences = [(evidences_list[i], predictions[i][0]) for i in range(len(predictions)) if predictions[i] > threshold]

print("Related Evidences:")
for ev, score in related_evidences:
    print(f"Evidence: {ev} \n\tScore: {score}")


Related Evidences:
Evidence: In his first paper on the matter, he estimated that global temperature would rise by around 5 to 6 °C (9.0 to 10.8 °F) if the quantity of CO 2 was doubled 
	Score: 0.512722909450531
Evidence: The 1990 IPCC First Assessment Report estimated that equilibrium climate sensitivity to a dou- bling of CO 2 lay between 1.5 and 4.5 °C (2.7 and 8.1 °F), with a "best guess in the light of current knowledge" of 2.5 °C (4.5 °F). 
	Score: 0.512722909450531
Evidence: John Bennet Lawes, English entrepreneur and agricultural scientist 
	Score: 0.512722909450531
